In [2]:
# ============================================================
# YouTube Data Collector
# K-pop Sentiment Analysis Project
# Collects engagement metrics and date-filtered comments
# ============================================================

import time
import pandas as pd
from datetime import datetime, timezone
from googleapiclient.discovery import build

# --- API Key ---
with open("../credentials/api_keys.txt", "r") as f:
    for line in f:
        if line.startswith("YOUTUBE_API_KEY"):
            API_KEY = line.strip().split("=")[1]

# --- Video IDs ---
VIDEOS = {
    "aespa_Whiplash":           "jWQx2f-CErU",
    "IVE_RebelHeart":           "g36q0ZLvygQ",
    "TWICE_Strategy":           "Sz_wWzgh-vQ",
    "NCTDREAM_WhenImWithYou":   "B1qq8IvzSz4",
    "ATEEZ_IceOnMyTeeth":       "5OflOlcHLb8",
    "StrayKids_ChkChkBoom":     "0P0aQreFs8w"
}

# --- 14-day comeback windows ---
COMEBACK_WINDOWS = {
    "aespa_Whiplash":           ("2024-10-21", "2024-11-04"),
    "IVE_RebelHeart":           ("2025-01-13", "2025-01-27"),
    "TWICE_Strategy":           ("2024-12-06", "2024-12-20"),
    "NCTDREAM_WhenImWithYou":   ("2024-11-11", "2024-11-25"),
    "ATEEZ_IceOnMyTeeth":       ("2024-11-15", "2024-11-29"),
    "StrayKids_ChkChkBoom":     ("2024-07-19", "2024-08-02")
}

# --- Build YouTube client ---
youtube = build("youtube", "v3", developerKey=API_KEY)

# ============================================================
# PART 1: Collect engagement metrics (current totals)
# ============================================================
print("=" * 50)
print("PART 1: Collecting engagement metrics")
print("=" * 50)

metrics = []
for name, video_id in VIDEOS.items():
    request = youtube.videos().list(part="statistics,snippet", id=video_id)
    response = request.execute()
    if response["items"]:
        item = response["items"][0]
        stats = item["statistics"]
        snippet = item["snippet"]
        metrics.append({
            "group_comeback": name,
            "video_id": video_id,
            "title": snippet["title"],
            "published_at": snippet["publishedAt"],
            "view_count": int(stats.get("viewCount", 0)),
            "like_count": int(stats.get("likeCount", 0)),
            "comment_count": int(stats.get("commentCount", 0)),
            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })
        print(f"Collected metrics: {name}")

metrics_df = pd.DataFrame(metrics)
print(metrics_df[["group_comeback", "view_count", "like_count", "comment_count"]])
metrics_df.to_csv("../07_tableau/youtube_metrics_snapshot.csv", index=False)
print("Saved: 07_tableau/youtube_metrics_snapshot.csv")

# ============================================================
# PART 2: Collect comments from 14-day comeback window
# ============================================================
print("\n" + "=" * 50)
print("PART 2: Collecting comments from comeback windows")
print("=" * 50)

def get_comments_by_date(youtube, video_id, start_date, end_date, max_comments=500):
    comments = []
    next_page_token = None
    start_dt = datetime.strptime(start_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    end_dt = datetime.strptime(end_date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    stop_collecting = False

    while len(comments) < max_comments and not stop_collecting:
        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=100,
            pageToken=next_page_token,
            order="time",
            textFormat="plainText"
        )
        response = request.execute()

        for item in response["items"]:
            comment = item["snippet"]["topLevelComment"]["snippet"]
            published = datetime.strptime(
                comment["publishedAt"], "%Y-%m-%dT%H:%M:%SZ"
            ).replace(tzinfo=timezone.utc)

            if published > end_dt:
                continue
            if published < start_dt:
                stop_collecting = True
                break

            comments.append({
                "video_id": video_id,
                "comment": comment["textDisplay"],
                "likes": comment["likeCount"],
                "published_at": comment["publishedAt"]
            })

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break
        time.sleep(0.5)

    return comments

all_comments = []
for name, video_id in VIDEOS.items():
    start_date, end_date = COMEBACK_WINDOWS[name]
    print(f"Collecting comments for {name} ({start_date} to {end_date})...")
    comments = get_comments_by_date(youtube, video_id, start_date, end_date)
    for c in comments:
        c["group_comeback"] = name
    all_comments.extend(comments)
    print(f"  Collected {len(comments)} comments")
    time.sleep(1)

comments_df = pd.DataFrame(all_comments)
print(f"\nTotal comments: {len(comments_df)}")
print(comments_df.groupby("group_comeback").size())
comments_df.to_csv("../01_raw_data/youtube/youtube_comments_raw.csv", index=False)
print("Saved: 01_raw_data/youtube/youtube_comments_raw.csv")

PART 1: Collecting engagement metrics
Collected metrics: aespa_Whiplash
Collected metrics: IVE_RebelHeart
Collected metrics: TWICE_Strategy
Collected metrics: NCTDREAM_WhenImWithYou
Collected metrics: ATEEZ_IceOnMyTeeth
Collected metrics: StrayKids_ChkChkBoom
           group_comeback  view_count  like_count  comment_count
0          aespa_Whiplash   272860242     2496666          94955
1          IVE_RebelHeart    64041208      784179          42838
2          TWICE_Strategy   146374901     1970932         204572
3  NCTDREAM_WhenImWithYou    15889568      515104          42525
4      ATEEZ_IceOnMyTeeth    98921178     1062250          68104
5    StrayKids_ChkChkBoom   198946095     3631709         391720
Saved: 07_tableau/youtube_metrics_snapshot.csv

PART 2: Collecting comments from comeback windows
  Collected 565 comments
  Collected 576 comments
  Collected 523 comments
  Collected 555 comments
  Collected 526 comments
  Collected 538 comments

Total comments: 3283
group_comeback


In [1]:
import pandas as pd

df = pd.read_csv("../01_raw_data/youtube/youtube_comments_raw.csv")
df["published_at"] = pd.to_datetime(df["published_at"])
print(df.groupby("group_comeback")["published_at"].agg(["min", "max"]))

                                             min                       max
group_comeback                                                            
ATEEZ_IceOnMyTeeth     2024-11-27 08:34:22+00:00 2024-11-28 23:59:28+00:00
IVE_RebelHeart         2025-01-26 04:24:19+00:00 2025-01-26 23:47:54+00:00
NCTDREAM_WhenImWithYou 2024-11-15 03:46:40+00:00 2024-11-24 23:55:56+00:00
StrayKids_ChkChkBoom   2024-07-19 04:48:18+00:00 2024-08-01 23:58:11+00:00
TWICE_Strategy         2024-12-19 08:12:54+00:00 2024-12-19 23:59:05+00:00
aespa_Whiplash         2024-11-02 14:16:59+00:00 2024-11-03 23:56:38+00:00
